# Quick Start

A ten minute tour of what you use every day: build a path, connect to S3, write and read an object, list a folder, copy and delete. Each section ends with the chapter that covers it in depth.

## Setup

This cell only prepares the notebook: it connects to the S3 bucket that the s3pathlib CI uses and empties the folder of this chapter, so every run starts from the same state. In your own code you only need the two lines marked below, with your own AWS profile and bucket.

In [1]:
import os

import boto3
from s3pathlib.api import S3Path
from s3pathlib.api import s3_context

boto_ses = boto3.Session(profile_name=os.environ["S3PATHLIB_TEST_AWS_PROFILE"])
s3_context.attach_s3_client(boto_ses.client("s3"))  # <-- in your code
aws_account_id = boto_ses.client("sts").get_caller_identity()["Account"]
bucket = f"{aws_account_id}-{boto_ses.region_name}-s3pathlib-test"  # <-- your bucket

# every demo of this chapter writes under this folder
s3dir_chapter = S3Path(bucket, "projects/s3pathlib/docs/01-Quick-Start/")
_ = s3dir_chapter.delete()

## 1. Build a path

An `S3Path` is a bucket plus a key, like `pathlib.Path` is a drive plus a path. Building one never calls AWS. The first argument is the bucket (or an S3 URI / ARN), the others are parts of the key.

In [2]:
p = S3Path("my-bucket", "folder", "file.txt")
p

S3Path('s3://my-bucket/folder/file.txt')

In [3]:
print(p.bucket)
print(p.key)
print(p.uri)
print(p.arn)
print(p.basename)
print(p.parent)

my-bucket
folder/file.txt
s3://my-bucket/folder/file.txt
arn:aws:s3:::my-bucket/folder/file.txt
file.txt
S3Path('s3://my-bucket/folder/')


A trailing `/` makes a folder. S3 itself has no folders, only keys; s3pathlib treats every key that ends with `/` as a folder, the same as the AWS console.

In [4]:
S3Path("s3://my-bucket/folder/file.txt").is_file(), S3Path("my-bucket", "folder/").is_dir()

(True, True)

Join with `/`. Every part on the left of `/` must be a folder, so it ends with `/`:

In [5]:
S3Path("my-bucket") / "folder/" / "file.txt"

S3Path('s3://my-bucket/folder/file.txt')

More in [Path Manipulation](../02-Path-Manipulation/index.ipynb).

## 2. Connect to S3

Methods that call AWS use the S3 client attached to `s3_context` (the setup cell above did that). Without one, s3pathlib creates `boto3.client("s3")` with your default credentials the first time it needs one. Every such method also takes `s3_client=...` for a single call, e.g. to work with two AWS accounts. More in [Clients and Errors](../10-Clients-and-Errors/index.ipynb).

## 3. Write and read

The demos below each use their own folder under `s3dir_chapter`.

In [6]:
s3dir = s3dir_chapter / "write-and-read/"
p = s3dir / "hello.txt"
p.exists()

False

`write_text` (and `write_bytes`) returns a **new** `S3Path` that already knows the size and the ETag of what it wrote, so reading them costs no extra call.

In [7]:
p_new = p.write_text("Hello, S3!")
p_new.size, p_new.etag

(10, '9176e88e2973b7e2e260c438f56b1cd0')

In [8]:
p.exists(), p.read_text()

(True, 'Hello, S3!')

More in [Read and Write](../03-Read-and-Write/index.ipynb): bytes, conditional writes, file objects with `open`, uploading local files.

## 4. List a folder

`iter_objects` lists every object under a folder (recursively by default), in the order S3 returns them, sorted by key.

In [9]:
s3dir = s3dir_chapter / "list/"
for name in ["a.txt", "b.csv", "sub/c.csv"]:
    _ = (s3dir / name).write_text(name)

s3dir.iter_objects().all()

[S3Path('s3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/01-Quick-Start/list/a.txt'),
 S3Path('s3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/01-Quick-Start/list/b.csv'),
 S3Path('s3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/01-Quick-Start/list/sub/c.csv')]

The full URIs are long; from here on the demos show each path relative to its folder (`p - s3dir` is `p.relative_to(s3dir)`). `iterdir` lists one level, folders included, like `pathlib.Path.iterdir`:

In [10]:
[p - s3dir for p in s3dir.iterdir()]

[S3RelPath('a.txt'), S3RelPath('b.csv'), S3RelPath('sub/')]

The result can be filtered by extension, or by any attribute such as `size`; each path already has the `size`, `etag` and `last_modified_at` returned by the list call:

In [11]:
[p - s3dir for p in s3dir.iter_objects().filter_by_ext(".csv")]

[S3RelPath('b.csv'), S3RelPath('sub/c.csv')]

In [12]:
s3dir.count_objects(), s3dir.calculate_total_size()

(3, (3, 19))

More in [List and Filter](../04-List-and-Filter/index.ipynb).

## 5. Copy

`copy_file` copies one object and returns the new `S3Path` of the copy. It does not replace an existing object unless you pass `overwrite=True`.

In [13]:
s3dir = s3dir_chapter / "copy/"
src = s3dir / "src.txt"
_ = src.write_text("some data")
dst = s3dir / "dst.txt"

copied = src.copy_file(dst)
copied, dst.read_text()

(S3Path('s3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/01-Quick-Start/copy/dst.txt'),
 'some data')

In [14]:
from s3pathlib.api import exc

try:
    src.copy_file(dst)
except exc.S3FileAlreadyExist as e:
    print(f"{type(e).__name__}: {e}")

S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/01-Quick-Start/copy/dst.txt' already exists!


`copy_dir` copies every object under a folder and returns how many it copied:

In [15]:
(s3dir_chapter / "list/").copy_dir(s3dir / "backup/")

3

In [16]:
[p - s3dir for p in s3dir.iter_objects()]

[S3RelPath('backup/a.txt'),
 S3RelPath('backup/b.csv'),
 S3RelPath('backup/sub/c.csv'),
 S3RelPath('dst.txt'),
 S3RelPath('src.txt')]

More in [Copy, Move and Delete](../06-Copy-Move-Delete/index.ipynb).

## 6. Delete

`delete` deletes an object, or every object under a folder. Deleting what does not exist is not an error.

In [17]:
p = s3dir_chapter / "delete/" / "file.txt"
_ = p.write_text("bye")
_ = p.delete()
p.exists()

False

In [18]:
s3dir = s3dir_chapter / "copy/"
_ = s3dir.delete()
s3dir.exists(), s3dir.count_objects()

(False, 0)

In a bucket with versioning, `delete` adds a delete marker and keeps the old versions; see [Versioning](../07-Versioning/index.ipynb).

## Where to go next

- [Path Manipulation](../02-Path-Manipulation/index.ipynb): everything you can do with a path without calling AWS.
- [Read and Write](../03-Read-and-Write/index.ipynb), [List and Filter](../04-List-and-Filter/index.ipynb), [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb), [Copy, Move and Delete](../06-Copy-Move-Delete/index.ipynb): the S3 operations.
- [Versioning](../07-Versioning/index.ipynb), [Buckets](../08-Buckets/index.ipynb), [Sync](../09-Sync/index.ipynb), [Clients and Errors](../10-Clients-and-Errors/index.ipynb).
- [Upgrade from 2.x](../11-Upgrade-from-2.x/index.ipynb) if you used s3pathlib before 3.0.0.